# NB01 — Regulation Library Parsing

**Project:** Scientific AI Builder — Build02 Regulation Library  
**Goal:** Parse the five EUR-Lex HTML files into a structured form before chunking and embedding.

This notebook first verifies that **the original legal documents are parsed correctly**, before building a RAG pipeline.

### What this notebook checks
1. File loading
2. Legal-document metadata extraction
3. Article-level extraction
4. HTML table-row extraction
5. Whether key benchmark information can be found in the parsed content:
   - `Dried figs`
   - `Türkiye`
   - `Article 19`
   - `Prior notification`

> **Important:** This stage uses neither embeddings nor an LLM.  
> Reliable source parsing must be verified first.


## 0. Setup

The following cell imports the required libraries:

- `BeautifulSoup` — HTML parsing
- `pandas` — inspecting results
- `pathlib` — managing file paths


In [1]:
# Resolve paths from the repository root, including nested Jupyter kernel CWDs.
import os as _repo_os
from pathlib import Path as _RepoPath
_repo_start = _RepoPath(_repo_os.environ.get("REGULATORY_AGENT_ROOT", _RepoPath.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_repo_start, *_repo_start.parents)
                  if (p / "data/benchmark/benchmark_questions_v1.json").is_file()
                  and (p / "notebooks/build04/NB09_v3.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REGULATORY_AGENT_ROOT to the Regulatory_agent checkout.")
_repo_os.chdir(REPO_ROOT)

# If needed, uncomment this line:
# !pip install beautifulsoup4 lxml pandas

from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re
from collections import Counter

pd.set_option("display.max_colwidth", 160)


## 1. Regulation Library

The five EUR-Lex documents used in Build02 are stored in `data/legal_sources/`.

The setup cell resolves the repository root before the relative paths below are used. The document paths are constructed explicitly from `data/legal_sources/`; changing `BASE_DIR` alone does not redirect them.


In [2]:
# Change this path if you run the notebook on your own computer.
BASE_DIR = Path("data")

FILES = [
    "EU_2023_915_consolidated_2026-08-19.html",
    "EU_2002_178_consolidated_2026-01-01.html",
    "EU_2017_625_consolidated_2025-01-05.html",
    "EU_2019__1013_original.html",
    "EU_2019_1793_consolidated_2026-06-30.html",
]

paths = [__import__("pathlib").Path("data/legal_sources") / f for f in FILES]

for p in paths:
    print(f"{'✓' if p.exists() else '✗'}  {p}")


✓  data\EU_2023_915_consolidated_2026-08-19.html
✓  data\EU_2002_178_consolidated_2026-01-01.html
✓  data\EU_2017_625_consolidated_2025-01-05.html
✓  data\EU_2019__1013_original.html
✓  data\EU_2019_1793_consolidated_2026-06-30.html


### Repository paths and external notebook kernels

Run the setup cell first. It locates the repository root from the kernel's working directory or its parents, so kernels started inside `notebooks/build01_02/` are supported.

If the kernel starts outside the checkout, set the `REGULATORY_AGENT_ROOT` environment variable to the absolute checkout path before running setup. The notebook uses `data/legal_sources/` for the frozen HTML snapshots and `data/processed/` for parsed artifacts. The earlier temporary-workspace `/mnt/data` and `data/regulations/` suggestions do not describe the current layout.


## 2. Helper functions

EUR-Lex consolidated HTML includes more than the legal text:

- amendment history
- Official Journal references
- scripts and styles
- legal text (`eli-container`)
- Articles
- Annex tables

Rather than relying on a single `get_text()` call, the parser **extracts metadata, Articles and table rows separately**.


In [3]:
def clean_text(text: str) -> str:
    """Collapse repeated whitespace while preserving readable text."""
    if text is None:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def infer_doc_type_and_date(filename: str, title: str):
    """Infer whether the document is consolidated/original and its version date."""
    low = filename.lower()
    if "consolidated" in low or "consolidated text" in title.lower():
        doc_type = "consolidated"
    else:
        doc_type = "original"

    # Prefer YYYY-MM-DD from filename.
    m = re.search(r"(20\d{2}-\d{2}-\d{2})", filename)
    version_date = m.group(1) if m else None

    # If no ISO date is in filename, try EUR-Lex title format DD.MM.YYYY.
    if not version_date:
        m2 = re.search(r"(\d{2})\.(\d{2})\.(20\d{2})", title)
        if m2:
            version_date = f"{m2.group(3)}-{m2.group(2)}-{m2.group(1)}"

    return doc_type, version_date


def parse_eurlex_html(path: Path):
    """Parse one EUR-Lex HTML file into metadata, articles, and table rows."""
    html = path.read_text(encoding="utf-8", errors="replace")
    soup = BeautifulSoup(html, "lxml")

    title = clean_text(soup.title.get_text(" ", strip=True) if soup.title else "")
    canonical_tag = soup.find("link", rel="canonical")
    canonical_url = canonical_tag.get("href") if canonical_tag else None

    reference = None
    ref_tag = soup.find("p", class_="reference")
    if ref_tag:
        reference = clean_text(ref_tag.get_text(" ", strip=True))

    # Regulation title: consolidated pages usually use title-doc-first;
    # original OJ pages may use oj-doc-ti.
    regulation_title = None
    for selector in [
        "div.eli-main-title p.title-doc-first",
        "div.eli-main-title p.oj-doc-ti",
        "p.title-doc-first",
        "p.oj-doc-ti",
    ]:
        tag = soup.select_one(selector)
        if tag:
            regulation_title = clean_text(tag.get_text(" ", strip=True))
            if regulation_title:
                break

    doc_type, version_date = infer_doc_type_and_date(path.name, title)

    # ---------- Article extraction ----------
    articles = []
    article_divs = []

    # Most consolidated EUR-Lex pages:
    for div in soup.find_all("div", class_="eli-subdivision"):
        div_id = div.get("id", "")
        if re.fullmatch(r"art_\d+", div_id):
            article_divs.append(div)

    for div in article_divs:
        art_id = div.get("id")

        title_tag = (
            div.find("p", class_=re.compile(r"title-article"))
            or div.find("p", class_="oj-ti-art")
        )
        article_label = clean_text(title_tag.get_text(" ", strip=True)) if title_tag else art_id

        heading_tag = div.find("div", class_="eli-title")
        heading = clean_text(heading_tag.get_text(" ", strip=True)) if heading_tag else ""

        text = clean_text(div.get_text(" ", strip=True))

        articles.append({
            "article_id": art_id,
            "article": article_label,
            "heading": heading,
            "text": text,
        })

    # ---------- Table row extraction ----------
    table_rows = []
    for t_idx, table in enumerate(soup.find_all("table"), start=1):
        for r_idx, tr in enumerate(table.find_all("tr"), start=1):
            cells = [
                clean_text(td.get_text(" ", strip=True))
                for td in tr.find_all(["th", "td"], recursive=False)
            ]
            cells = [c for c in cells if c]
            if len(cells) >= 2:
                table_rows.append({
                    "table_index": t_idx,
                    "row_index": r_idx,
                    "cells": cells,
                    "row_text": " | ".join(cells),
                })

    # Legal body text — use the main ELI container if present.
    containers = soup.find_all("div", class_="eli-container")
    if containers:
        body_text = clean_text(" ".join(c.get_text(" ", strip=True) for c in containers))
    else:
        body_text = clean_text(soup.get_text(" ", strip=True))

    return {
        "file": path.name,
        "path": str(path),
        "document_type": doc_type,
        "version_date": version_date,
        "title": title,
        "regulation_title": regulation_title,
        "reference": reference,
        "canonical_url": canonical_url,
        "articles": articles,
        "table_rows": table_rows,
        "body_text": body_text,
    }


## 3. Parse all five documents

Read the five HTML files into structured Python objects.


In [4]:
docs = []

for path in paths:
    if not path.exists():
        print(f"SKIP - file not found: {path}")
        continue
    docs.append(parse_eurlex_html(path))

print(f"Parsed documents: {len(docs)}")


C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_21064\3082640097.py:32: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  soup = BeautifulSoup(html, "lxml")


Parsed documents: 5


## 4. Metadata check

First, verify that **the intended document versions were downloaded**.

In particular:
- 2023/915 → consolidated 2026-08-19
- 2019/1793 → consolidated 2026-06-30
- 2019/1013 → original


In [5]:
metadata_df = pd.DataFrame([
    {
        "file": d["file"],
        "type": d["document_type"],
        "version_date": d["version_date"],
        "regulation_title": d["regulation_title"],
        "reference": d["reference"],
        "canonical_url": d["canonical_url"],
        "articles": len(d["articles"]),
        "table_rows": len(d["table_rows"]),
    }
    for d in docs
])

metadata_df


,file,type,version_date,regulation_title,reference,canonical_url,articles,table_rows
0,EU_2023_915_consolidated_2026-08-19.html,consolidated,2026-08-19,COMMISSION REGULATION (EU) 2023/915,02023R0915 — EN — 19.08.2026 — 007.001,https://eur-lex.europa.eu/eli/reg/2023/915/2026-08-19/eng,11,590
1,EU_2002_178_consolidated_2026-01-01.html,consolidated,2026-01-01,REGULATION (EC) No 178/2002 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL,02002R0178 — EN — 01.01.2026 — 012.001,https://eur-lex.europa.eu/eli/reg/2002/178/2026-01-01/eng,65,13
2,EU_2017_625_consolidated_2025-01-05.html,consolidated,2025-01-05,REGULATION (EU) 2017/625 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL,02017R0625 — EN — 05.01.2025 — 004.001,https://eur-lex.europa.eu/eli/reg/2017/625/2025-01-05/eng,167,751
3,EU_2019__1013_original.html,original,None,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1013,None,https://eur-lex.europa.eu/eli/reg_impl/2019/1013/oj/eng,2,7
4,EU_2019_1793_consolidated_2026-06-30.html,consolidated,2026-06-30,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1793,02019R1793 — EN — 30.06.2026 — 015.001,https://eur-lex.europa.eu/eli/reg_impl/2019/1793/2026-06-30/eng,15,501


### Expected interpretation

An empty `reference` field does not necessarily indicate an error.  
For example, original Official Journal HTML may have a different structure from a consolidated document.

Check that the following fields are extracted plausibly:

- `regulation_title`
- `version_date`
- `articles`
- `canonical_url`


## 5. Inspect extracted Articles

Verify that the Article structure is preserved in the parsed records.


In [6]:
article_summary = []

for d in docs:
    for a in d["articles"]:
        article_summary.append({
            "file": d["file"],
            "article_id": a["article_id"],
            "article": a["article"],
            "heading": a["heading"],
            "preview": a["text"][:220],
        })

articles_df = pd.DataFrame(article_summary)
articles_df.head(30)


,file,article_id,article,heading,preview
0,EU_2023_915_consolidated_2026-08-19.html,art_1,Article 1,Definitions,"Article 1 Definitions For the purposes of this Regulation, the following definitions apply: (a) ‘food’ means food as defined in Article 2 of Regulation (EC)..."
1,EU_2023_915_consolidated_2026-08-19.html,art_2,Article 2,General rules,Article 2 General rules 1. The food listed in Annex I shall not be placed on the market and shall not be used as a raw material in food or as an ingredient ...
2,EU_2023_915_consolidated_2026-08-19.html,art_3,Article 3,"Dried, diluted, processed and compound food","Article 3 Dried, diluted, processed and compound food 1. Where no specific Union maximum levels are set out in Annex I for food which is dried, diluted, pro..."
3,EU_2023_915_consolidated_2026-08-19.html,art_4,Article 4,Prohibition on detoxification,Article 4 Prohibition on detoxification Food containing contaminants listed in Annex I shall not be deliberately detoxified by chemical treatments.
4,EU_2023_915_consolidated_2026-08-19.html,art_5,Article 5,Food to be subjected to sorting or other physical treatment before placing on the market for the final consumer or use as a food ingredient,Article 5 Food to be subjected to sorting or other physical treatment before placing on the market for the final consumer or use as a food ingredient 1. Whe...
5,EU_2023_915_consolidated_2026-08-19.html,art_6,Article 6,"Labelling provisions for groundnuts (peanuts), other oilseeds, derived products thereof and cereals","Article 6 Labelling provisions for groundnuts (peanuts), other oilseeds, derived products thereof and cereals 1. The label of each individual package and th..."
6,EU_2023_915_consolidated_2026-08-19.html,art_7,Article 7,Derogations from Article 2,"Article 7 Derogations from Article 2 1. By way of derogation from Article 2, Latvia, Finland and Sweden may authorise the placing on their respective market..."
7,EU_2023_915_consolidated_2026-08-19.html,art_8,Article 8,Monitoring and reporting,"Article 8 Monitoring and reporting 1. By 1 July 2023, Member States and interested parties shall communicate to the Commission the results of investigations..."
8,EU_2023_915_consolidated_2026-08-19.html,art_9,Article 9,Repeal,Article 9 Repeal Regulation (EC) No 1881/2006 is repealed. References to the repealed Regulation shall be construed as references to this Regulation and sha...
9,EU_2023_915_consolidated_2026-08-19.html,art_10,Article 10,Transitional measures,Article 10 Transitional measures 1. ►M11 Food lawfully placed on the market prior to the dates referred to in points (a) to (r) may remain on the market unt...


## 6. Search helper

Before building RAG, establish:

> **Does the parsed content actually contain the information we need to retrieve?**

The functions below search both Article text and table rows.


In [7]:
def search_library(query: str, docs, case_sensitive=False):
    """Search both articles and extracted table rows."""
    needle = query if case_sensitive else query.lower()
    results = []

    for d in docs:
        # Article search
        for a in d["articles"]:
            hay = a["text"] if case_sensitive else a["text"].lower()
            if needle in hay:
                results.append({
                    "file": d["file"],
                    "source_type": "article",
                    "location": f'{a["article"]} {a["heading"]}'.strip(),
                    "text": a["text"],
                })

        # Table-row search
        for row in d["table_rows"]:
            hay = row["row_text"] if case_sensitive else row["row_text"].lower()
            if needle in hay:
                results.append({
                    "file": d["file"],
                    "source_type": "table_row",
                    "location": f'Table {row["table_index"]}, row {row["row_index"]}',
                    "text": row["row_text"],
                })

        # Fallback body-level signal (useful if HTML structure differs)
        hay = d["body_text"] if case_sensitive else d["body_text"].lower()
        if needle in hay and not any(r["file"] == d["file"] for r in results):
            results.append({
                "file": d["file"],
                "source_type": "body_only",
                "location": "full legal body",
                "text": f"[Found '{query}' in full body, but not in extracted article/table rows]",
            })

    return pd.DataFrame(results)


def show_search(query, max_rows=20):
    print(f"SEARCH: {query!r}")
    df = search_library(query, docs)
    if df.empty:
        print("No matches.")
        return df
    display(df.head(max_rows))
    return df


## 7. Diagnostic test 1 — `Dried figs`

**Expected:** The search should return dried-fig rows from the Annex tables of Regulation (EU) 2023/915.

> Check whether the numerical limits and product name appear together in the same table row.


In [8]:
dried_figs_results = show_search("Dried figs")


SEARCH: 'Dried figs'


,file,source_type,location,text
0,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 3, row 6","1.1.3 | Dried figs | 6,0 | 10,0 | - | In the case of food consisting of dried figs used as only ingredient or in the case of processed products consisting a..."
1,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 4, row 3","1.2.1.1 | Dried vine fruits (currants, raisins and sultanas) and dried figs | 8,0"
2,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 4, row 18","1.2.11.2 | products containing at least 20 % dried vine fruits and/or dried figs | 4,0"
3,EU_2019_1793_consolidated_2026-06-30.html,table_row,"Table 5, row 135",14 | Türkiye (TR) | — Dried figs | 0804 20 90
4,EU_2019_1793_consolidated_2026-06-30.html,table_row,"Table 5, row 142","— Dried figs, prepared or preserved, including mixtures | ex 2008 97 12 | 11"
5,EU_2019_1793_consolidated_2026-06-30.html,table_row,"Table 5, row 169","— Flours, meals and powder of dried figs | ex 1106 30 90 | 60"


## 8. Diagnostic test 2 — `Türkiye`

**Expected:** The search should return table rows associated with Annex II of Regulation (EU) 2019/1793.

Where possible, inspect the same row or neighboring rows for:

- Türkiye
- dried figs
- aflatoxins
- ochratoxin A
- check frequency


In [9]:
turkiye_results = show_search("Türkiye")


SEARCH: 'Türkiye'


,file,source_type,location,text
0,EU_2019_1793_consolidated_2026-06-30.html,table_row,"Table 4, row 192","24 | Türkiye (TR) | Lemons ( Citrus limon , Citrus limonum ) ( Food – fresh, chilled or dried ) | 0805 50 10 | Pesticide residues ( 3 ) | 20"
1,EU_2019_1793_consolidated_2026-06-30.html,table_row,"Table 5, row 135",14 | Türkiye (TR) | — Dried figs | 0804 20 90
2,EU_2019_1793_consolidated_2026-06-30.html,table_row,"Table 6, row 2","1 | United States (US) | Türkiye (TR) ( 2 ) | — Pistachios, in shell | 0802 51 00 | Aflatoxins | 50"


## 9. Diagnostic test 3 — `Article 19`

Check whether the withdrawal, recall and notification responsibilities in Regulation (EC) No 178/2002 are available for later RAG retrieval.

Inspecting the extracted Article 19 structure directly is more precise here than simply searching for the string `"Article 19"`.


In [10]:
article_19_rows = articles_df[
    articles_df["article"].str.replace("\xa0", " ", regex=False).str.match(r"Article\s+19\b", na=False)
]

article_19_rows


,file,article_id,article,heading,preview
29,EU_2002_178_consolidated_2026-01-01.html,art_19,Article 19,Responsibilities for food: food business operators,Article 19 Responsibilities for food: food business operators 1. If a food business operator considers or has reason to believe that a food which it has imp...
94,EU_2017_625_consolidated_2025-01-05.html,art_19,Article 19,Specific rules on official controls and for action taken by the competent authorities in relation to the residues of relevant substances in food and feed,Article 19 Specific rules on official controls and for action taken by the competent authorities in relation to the residues of relevant substances in food ...


## 10. Diagnostic test 4 — `Prior notification`

Verify that the prior-notification provisions are extracted correctly from the original HTML of Regulation (EU) 2019/1013.


In [11]:
prior_notification_results = show_search("Prior notification")


SEARCH: 'Prior notification'


,file,source_type,location,text
0,EU_2017_625_consolidated_2025-01-05.html,article,Article 56 Use of the Common Health Entry Document (CHED) by the operator and by the competent authorities,Article 56 Use of the Common Health Entry Document (CHED) by the operator and by the competent authorities 1. For each consignment of the categories of anim...
1,EU_2017_625_consolidated_2025-01-05.html,article,"Article 58 Format, time requirements and specific rules for the use of the CHED","Article 58 Format, time requirements and specific rules for the use of the CHED The Commission shall, by means of implementing acts, lay down rules on: (a) ..."
2,EU_2019__1013_original.html,article,Article 1 Prior notification of consignments,Article 1 Prior notification of consignments 1. The operator responsible for a consignment falling within the categories of animals and goods referred to in...
3,EU_2019__1013_original.html,table_row,"Table 3, row 1",(2) | Regulation (EU) 2017/625 requires the operator responsible for certain consignments entering the Union to give prior notification to the competent aut...
4,EU_2019__1013_original.html,table_row,"Table 4, row 1","(3) | However, due to transport-related logistical constraints, it may not be possible to comply with the prior notification period of one working day befor..."
5,EU_2019__1013_original.html,table_row,"Table 5, row 1",(4) | Commission Delegated Regulation (EU) 2019/1012 ( 2 ) provides for Member States to exempt border control posts which are to be designated for imports ...


## 11. Inspect the full Article 1 of Regulation 2019/1013

Verify that **one working day / four hours**, which matter for Q5, are present in the parsed content.


In [12]:
for d in docs:
    if "1013" in d["file"]:
        for a in d["articles"]:
            if re.search(r"Article\s+1\b", a["article"].replace("\xa0", " ")):
                print(a["article"], "-", a["heading"])
                print()
                print(a["text"])


Article 1 - Prior notification of consignments

Article 1 Prior notification of consignments 1. The operator responsible for a consignment falling within the categories of animals and goods referred to in Article 47(1) of Regulation (EU) 2017/625 shall give prior notification, to the competent authority of the border control post of first arrival into the Union, at least one working day before the expected arrival of the consignment. 2. By way of derogation from paragraph 1, where logistical constraints prevent compliance with the time limit set out in that paragraph, the competent authorities of the border control posts may apply a period of prior notification of at least four hours before the expected arrival of the consignment. 3. By way of derogation from paragraph 1, the competent authorities of border control posts designated for imports of unprocessed logs and sawn and chipped wood in accordance with Article 4 of Commission Delegated Regulation (EU) 2019/1012 may apply a period 

## 12. Quick QA report

The following table summarizes the basic parsing status of each document.

**PASS criteria**
- Metadata is extracted.
- At least one Article is extracted.
- The legal body text is not empty.

The number of table rows depends on the document structure; a document does not need to contain a table row to pass.


In [13]:
qa_rows = []

for d in docs:
    qa_rows.append({
        "file": d["file"],
        "metadata_ok": bool(d["regulation_title"] or d["title"]),
        "articles_ok": len(d["articles"]) > 0,
        "body_ok": len(d["body_text"]) > 100,
        "article_count": len(d["articles"]),
        "table_row_count": len(d["table_rows"]),
        "body_chars": len(d["body_text"]),
    })

qa_df = pd.DataFrame(qa_rows)
qa_df


,file,metadata_ok,articles_ok,body_ok,article_count,table_row_count,body_chars
0,EU_2023_915_consolidated_2026-08-19.html,True,True,True,11,590,112667
1,EU_2002_178_consolidated_2026-01-01.html,True,True,True,65,13,120882
2,EU_2017_625_consolidated_2025-01-05.html,True,True,True,167,751,340183
3,EU_2019__1013_original.html,True,True,True,2,7,6082
4,EU_2019_1793_consolidated_2026-06-30.html,True,True,True,15,501,60381


## 13. Save parsed intermediate data (optional)

The intermediate JSON lets **NB02 — Chunking** reuse the parsed library without parsing the HTML again.

To reduce file size, the export excludes `body_text` and retains only:

- metadata
- articles
- table rows

The frozen artifact is `data/processed/parsed_regulation_library.json`. The following export cell writes to that path; use `tests/offline_smoke.py` from the repository root to validate reproduction in temporary storage without overwriting it.


In [14]:
import json

output = []

for d in docs:
    output.append({
        "file": d["file"],
        "document_type": d["document_type"],
        "version_date": d["version_date"],
        "regulation_title": d["regulation_title"],
        "reference": d["reference"],
        "canonical_url": d["canonical_url"],
        "articles": d["articles"],
        "table_rows": d["table_rows"],
    })

OUT_JSON = __import__("pathlib").Path("data/processed/parsed_regulation_library.json")

with OUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("Saved:", OUT_JSON)


Saved: data\parsed_regulation_library.json


# NB01 completion checklist

Use this checklist to review the parsing results:

- [ ] All five documents loaded
- [ ] 2023/915 version = 2026-08-19
- [ ] 2019/1793 version = 2026-06-30
- [ ] Article structure extracted
- [ ] `Dried figs` search successful
- [ ] `Türkiye` search successful
- [ ] Regulation 178/2002 Article 19 extraction confirmed
- [ ] `Prior notification` search successful
- [ ] **one working day / four hours** confirmed in 2019/1013 Article 1
- [ ] `data/processed/parsed_regulation_library.json` available

---

## Next: NB02 — Chunking

NB02 uses the parsed data to compare:

1. **Naive fixed-size chunking**
2. **Legal-structure-aware chunking**
   - Article-level units
   - Annex section structure
   - Table rows with header context

The central experimental question for regulatory RAG is how to **preserve legal structure and the relationships within tables**, rather than simply splitting text into fixed blocks such as 500 characters.
